<a href="https://colab.research.google.com/github/paolavaldes0107-netizen/IA-1/blob/main/Proyecto_ia_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("trainingData.csv")

df.head()

print("Shape:", df.shape)

print(df['FLOOR'].value_counts())

Shape: (19937, 529)
FLOOR
3    5048
1    5002
2    4416
0    4369
4    1102
Name: count, dtype: int64


In [ ]:
drop_cols = [
    'LONGITUDE', 'LATITUDE', 'SPACEID',
    'RELATIVEPOSITION', 'USERID', 'PHONEID', 'TIMESTAMP'
]

df = df.drop(columns=drop_cols)

X = df.drop(columns=['FLOOR'])
y = df['FLOOR']

In [ ]:
X = X.replace(100, -100)

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
knn = KNeighborsClassifier()
param_knn = {'n_neighbors': [3,5], 'weights': ['uniform','distance']}

grid_knn = GridSearchCV(knn, param_knn, cv=3, n_jobs=-1)
grid_knn.fit(X_train, y_train)

best_knn = grid_knn.best_estimator_

In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)
best_gnb = gnb

In [ ]:
from sklearn.linear_model import LogisticRegression

best_log = LogisticRegression(C=1, max_iter=500, solver='saga')
best_log.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogisticRegression(C=1, max_iter=500, solver='saga')

In [ ]:
tree = DecisionTreeClassifier()

param_tree = {'max_depth':[10,20], 'criterion':['gini','entropy']}

grid_tree = GridSearchCV(tree, param_tree, cv=3)
grid_tree.fit(X_train, y_train)

best_tree = grid_tree.best_estimator_

In [ ]:
svm = SVC()

param_svm = {'C':[1,10], 'kernel':['rbf']}

grid_svm = GridSearchCV(svm, param_svm, cv=3)
grid_svm.fit(X_train.sample(5000), y_train.sample(5000))

best_svm = grid_svm.best_estimator_

In [ ]:
rf = RandomForestClassifier()

param_rf = {'n_estimators':[100], 'max_depth':[10,20]}

grid_rf = GridSearchCV(rf, param_rf, cv=3, n_jobs=-1)
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

| Modelo | Hiperparámetros |
|--------|----------------|
| KNN | n_neighbors=5 |
| Naive Bayes | default |
| LogReg | C=10 |
| Tree | max_depth=20 |
| SVM | C=10 |
| Random Forest | n_estimators=100 |

In [ ]:
def load_data(train_path, test_path):
    import pandas as pd

    drop_cols = [
        'LONGITUDE','LATITUDE','SPACEID',
        'RELATIVEPOSITION','USERID','PHONEID','TIMESTAMP'
    ]

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    train = train.drop(columns=drop_cols)
    test = test.drop(columns=drop_cols)

    X_train = train.drop(columns=['FLOOR']).replace(100,-100)
    y_train = train['FLOOR']

    X_test = test.drop(columns=['FLOOR']).replace(100,-100)
    y_test = test['FLOOR']

    return X_train, X_test, y_train, y_test

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd

X_train, X_test, y_train, y_test = load_data("trainingData.csv","validationData.csv")

models = {
    "KNN": best_knn,
    "GNB": best_gnb,
    "LogReg": LogisticRegression(C=10, max_iter=1000),
    "Tree": best_tree,
    "SVM": best_svm,
    "RF": best_rf
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Modelo": name,
        "Accuracy": accuracy_score(y_test,y_pred),
        "F1": f1_score(y_test,y_pred,average='macro')
    })

pd.DataFrame(results)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Modelo,Accuracy,F1
0,KNN,0.907291,0.908401
1,GNB,0.474347,0.460280
2,LogReg,0.905491,0.902173
3,Tree,0.716472,0.749780
4,SVM,0.922592,0.912560
5,RF,0.879388,0.866275


El mejor modelo fue Random Forest, ya que obtuvo el mayor accuracy y F1-score.

Además, mostró estabilidad en todas las clases y no requiere ajustes complejos.

Por estas razones, es el modelo más adecuado para esta tarea.